In [0]:
spark.sql("DROP TABLE IF EXISTS de_mini_project.raw.sales")
# Define new table with custom columns
spark.sql("""
CREATE TABLE IF NOT EXISTS de_mini_project.raw.sales (
    transaction_id STRING,
    date DATE,
    product_id STRING,
    customer_id STRING,
    store_id STRING,
    sold_qty INTEGER,
    unit_price DECIMAL(10,2)
)
""")

# Insert selected columns from source table
spark.sql("""
INSERT INTO de_mini_project.raw.sales (transaction_id, date, product_id, customer_id, store_id, sold_qty, unit_price)
SELECT 
    trxn_id_ AS transaction_id,
    COALESCE(
        try_to_date(_date_ref_, 'dd-MM-yyyy'),
        try_to_date(_date_ref_, 'MMM dd, yyyy'),
        try_to_date(_date_ref_, 'yyyy.MM.dd'),
        try_to_date(_date_ref_, 'dd-MMM-yy'),
        try_to_date(_date_ref_, 'yyyyMMdd'),
        try_to_date(_date_ref_, 'MM-dd-yyyy')
    ) AS date,
    prod_code_id AS product_id,
    cust_id_99 AS customer_id,
    store_loc_id AS store_id,
    qty_sold AS sold_qty,
    CASE 
        WHEN _unit_price_ LIKE '%$%' THEN 
            CAST(regexp_replace(_unit_price_, '[^0-9.]', '') AS DECIMAL(10,2))*93
        ELSE 
            CAST(regexp_replace(_unit_price_, '[^0-9.]', '') AS DECIMAL(10,2))
    END AS unit_price
FROM de_mini_project.azure_blob_storage.sales
""")

